# KernelMLP - a learned kernel $\kappa(x,y)$

*Walkthrough notebook for `neural_operators/layers/kernel_nn.py`.*

The graph-based operators (GNO, LNO's bases, MGNO) need the kernel $\kappa(x,y)$ as an actual *function*.
We represent it with a small neural network: it takes the features of a point pair (for example the
concatenated coordinates $[x,y]$) and outputs a **matrix** of size `dv_out x dv_in`.

Why a matrix? The operator's hidden state $v(y)$ is a vector with `dv_in` channels and the output has
`dv_out` channels, so the kernel must turn one vector into another. Giving every pair its own learned
matrix is what makes the integral operator more expressive than a single shared weight matrix $W$.

## Setup

The first cell makes the repository importable when this notebook is opened from `notebooks/source/`; the second holds the module's imports.

In [ ]:
# Make the repo root importable so `import neural_operators` works from notebooks/source/.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "neural_operators").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import torch.nn as nn

## The module

The constructor stacks `Linear + ReLU` layers and ends with a `Linear` producing `dv_in * dv_out` numbers.
`forward` reshapes that flat vector into a matrix per edge.

In [ ]:
class KernelMLP(nn.Module):
    """A feed-forward network mapping edge features (e.g. concat(x, y)) to a
    dv_out x dv_in matrix, flattened to a vector of size dv_out*dv_in. This lets the
    kernel operator apply a *different, learned* linear map for every pair (x, y),
    which is what makes the integral kernel operator (7) more expressive than a single
    shared matrix W."""

    def __init__(self, in_features, dv_in, dv_out, hidden=64, n_hidden_layers=2):
        super().__init__()
        self.dv_in = dv_in
        self.dv_out = dv_out

        layers = [nn.Linear(in_features, hidden), nn.ReLU()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.ReLU()]
        layers += [nn.Linear(hidden, dv_in * dv_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, edge_features):
        # edge_features: (E, in_features) -> (E, dv_out, dv_in)
        out = self.net(edge_features)
        return out.view(-1, self.dv_out, self.dv_in)

## Try it

Feed in a batch of 5 edges (pairs of 2D points) and check we get five 8x8 matrices.

In [ ]:
import torch

kernel = KernelMLP(in_features=4, dv_in=8, dv_out=8, hidden=32)
edges = torch.rand(5, 4)                   # 5 edges, each = concat(x, y) with x, y in R^2
kappa = kernel(edges)
print("kernel output:", tuple(kappa.shape), " -> (num_edges, dv_out, dv_in)")

In [ ]:
# Applying the kernel to a feature vector at the source node = one "message".
v_y = torch.randn(5, 8)                    # feature vector at the source node of each edge
messages = torch.einsum("eoi,ei->eo", kappa, v_y)
print("messages:", tuple(messages.shape))